# Reproducing the two final entries

This notebook rebuilds both final submissions of *Detector-Guided Fusion of a U-Net and YOLO11 Detectors for Filament Instance Segmentation* (Solar Filament Segmentation Challenge 2026) from the trained weights, using the repository's own scripts, the ones that made the submissions:

| Entry | Models | Validation PQ | Public leaderboard |
|---|---|---|---|
| **validated** | run-3 U-Net + detectors 1, 2, 4, trained on the 563 training observations | 0.421 (cross-fitted, 144 observations) | 0.37 |
| **all-data** | the same recipes on all 707 observations: 4 U-Net seeds averaged, 2 seeds of each detector | none (validation data in training) | 0.37 |

Pipeline per entry: limb fit and limb-darkening correction → U-Net logits with 8-way test-time augmentation → YOLO11-seg detections → merge across detectors → detector-guided fusion → submission CSV → comparison with the file that was submitted. Section 6 also re-scores the validated entry on the validation set.

**Inputs.** The competition data; the weights dataset (14 checkpoints, the two submitted CSVs and `weights_manifest.json`, 824 MB); the repository at commit `a4ecc24`, which the notebook clones.

**Runtime.** About 70 minutes on Kaggle with two T4 GPUs, validation included, measured on 6 October 2026: preprocessing 8 min, the 18 model jobs 48 min (about 13 min per U-Net on the test set, 2 min per detector), fusion and comparison 2 min, validation scoring 11 min. The outputs shown below are from that run.

Training is not rerun here; section 7 lists the exact command and Kaggle session time of every model.

In [1]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

NOTEBOOK_START = time.time()
REPO_URL = "https://github.com/ameyypawar/filament-seg"
#: The code this notebook was tested with. A Kaggle run clones it; a local run
#: uses the checkout the notebook sits in.
REPO_COMMIT = "a4ecc2480232304c622b04db78d989786343b524"
ON_KAGGLE = Path("/kaggle/input").is_dir()

if ON_KAGGLE:
    # Under /tmp: /kaggle/working is saved as the notebook's output and capped at 20 GB.
    REPO = Path("/tmp/filament-seg")
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", REPO_COMMIT], check=True)
    # Kaggle's image already has PyTorch, OpenCV, pandas, SciPy and usually
    # pycocotools; the rest at the versions the entries were made with.
    import importlib.util
    packages = ["segmentation-models-pytorch==0.5.0", "ultralytics==8.4.173"]
    if importlib.util.find_spec("pycocotools") is None:
        packages.append("pycocotools")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
else:
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent

head = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print(f"repository {REPO} at {head[:7]}")

repository /tmp/filament-seg at a4ecc24


## Inputs and options

On Kaggle, the competition data and the weights dataset are found wherever they are mounted. A local run reads them from `FILAMENT_DATA_ROOT` (default: the repository's `data/` folder, filled by `scripts/download_data.sh`) and `FILAMENT_WEIGHTS`. Everything the notebook writes goes under `WORK`, including its own preprocessing cache, so it never touches the repository's `data/cache` or `outputs/`. The logit caches take about 10 GB, so on Kaggle `WORK` is under `/tmp` and only the results (submission files, validation scores, logs) are copied to `/kaggle/working`.

In [2]:
def find_competition_data() -> Path:
    for train_images in sorted(Path("/kaggle/input").rglob("train_images")):
        root = train_images.parent.parent
        if (root / "test" / "test_images").is_dir():
            return root
    raise SystemExit("competition data not found: add filament-segmentation-2026 as an input")


def find_weights() -> Path:
    for manifest in sorted(Path("/kaggle/input").rglob("weights_manifest.json")):
        return manifest.parent
    raise SystemExit("weights not found: add the weights dataset as an input")


if ON_KAGGLE:
    DATA_ROOT, WEIGHTS = find_competition_data(), find_weights()
    WORK, RESULTS = Path("/tmp/reproduce"), Path("/kaggle/working")
else:
    DATA_ROOT = Path(os.environ.get("FILAMENT_DATA_ROOT", REPO / "data" / "MAGFiLO_1.0_Kaggle_2026"))
    WEIGHTS = Path(os.environ["FILAMENT_WEIGHTS"])
    WORK = Path(os.environ.get("REPRODUCE_WORK", REPO / "outputs" / "reproduce"))
    RESULTS = WORK

#: Entries to rebuild, and whether to re-score the validated entry on validation.
ENTRIES_TO_RUN = ["validated", "all_data"]
VALIDATE = os.environ.get("REPRODUCE_VALIDATE", "1") == "1"
#: Debug: only the first N test images (0 = all 180).
LIMIT = int(os.environ.get("REPRODUCE_LIMIT", "0"))
TAG = f"_first{LIMIT}" if LIMIT else ""

CACHE, LOGITS, DETECTIONS, LOGS = (WORK / name for name in ("cache", "logits", "detections", "logs"))
for folder in (CACHE, LOGITS, DETECTIONS, LOGS):
    folder.mkdir(parents=True, exist_ok=True)

# The repository reads where its data and outputs live from these, at import.
os.environ["FILAMENT_DATA_ROOT"] = str(DATA_ROOT)
os.environ["FILAMENT_OUTPUT_ROOT"] = str(WORK / "outputs")
os.environ["PYTHONUNBUFFERED"] = "1"
sys.path.insert(0, str(REPO))

import torch

GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(f"data:    {DATA_ROOT}\nweights: {WEIGHTS}\nwork:    {WORK}\nresults: {RESULTS}")
print(f"python {sys.version.split()[0]}, torch {torch.__version__}, "
      f"GPUs: {GPUS or ('Apple MPS' if torch.backends.mps.is_available() else 'none, CPU only')}")
print(f"entries: {ENTRIES_TO_RUN}, validate: {VALIDATE}, test images: {LIMIT or 'all'}")

data:    /kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026
weights: /kaggle/input/datasets/ameypawar123456789/filament-seg-weights
work:    /tmp/reproduce
results: /kaggle/working
python 3.13.15, torch 2.11.0+cu128, GPUs: ['Tesla T4', 'Tesla T4']
entries: ['validated', 'all_data'], validate: True, test images: all


## The two entries

File names are those in the weights dataset, whose `weights_manifest.json` records the training run each file came from. Every detector predicts at the image size it was trained at, and an entry's detectors are merged in the order listed. The fusion setting was chosen for the validated entry on all 144 validation observations (section 6 re-derives it) and reused unchanged for the all-data entry. Each file is checked against its recorded SHA-256 before anything runs.

In [3]:
import hashlib

ENTRIES = {
    "validated": {
        "unets": ["validated_unet_run3_last.pt"],
        "detectors": [("validated_det1_yolo11s_1024_best.pt", 1024),
                      ("validated_det2_yolo11m_1280_best.pt", 1280),
                      ("validated_det4_yolo11m_1536_last.pt", 1536)],
        "submitted": "submission_validated.csv",
    },
    "all_data": {
        "unets": ["all_data_unet_s0.pt", "all_data_unet_s1.pt",
                  "all_data_unet_s2.pt", "all_data_unet_s3.pt"],
        "detectors": [("all_data_det1_s0.pt", 1024), ("all_data_det1_s1.pt", 1024),
                      ("all_data_det2_s0.pt", 1280), ("all_data_det2_s1.pt", 1280),
                      ("all_data_det4_s2.pt", 1536), ("all_data_det4_s3.pt", 1536)],
        "submitted": "submission_all_data.csv",
    },
}

FUSION = dict(threshold=0.5, close_radius=3, grow=4, min_score=0.30, min_area=200,
              keep_unclaimed=0, unclaimed_gap=24)

SHA256 = {
    "validated_unet_run3_last.pt": "1d1dfd8042b44a198d52efb4c5b928abbf861cbe0e973a1ee77174ad19bf06f9",
    "validated_det1_yolo11s_1024_best.pt": "d61d42124926766d7e08f5da010adeed2d2eadb58de0c7538e87ba5857d358ef",
    "validated_det2_yolo11m_1280_best.pt": "bbfd9ffbc44d8d6415078455765db286af00dec61dc7449909d22fdd9a4773ca",
    "validated_det4_yolo11m_1536_last.pt": "d70eea1d195db1ea11055bfad906e8c4bc57fec16cd85c8248cfd737cb80af25",
    "all_data_unet_s0.pt": "91db147cdf63f2a363d285bef980a1b4f53194166af113910b72d0c264bfb513",
    "all_data_unet_s1.pt": "da7a637ef26c63b6a6992e58babb9cdd82ec74d591016e481681282cb4a9b075",
    "all_data_unet_s2.pt": "5c0062bf2e3dbb55eef4de81d61ae89d890d10cb90e323e90de10abcfffec239",
    "all_data_unet_s3.pt": "279c708ffd7ac8d684b5bd73f016f5e4e8f52068e9372f5291da9b13736edecd",
    "all_data_det1_s0.pt": "83f5f77deb8def2f3714f870f7c1b7377375ea0aa82e97f9a81644c42708fbde",
    "all_data_det1_s1.pt": "d90c90754553f4740f918ef5b4ca28810e3555ea8850c807daee81a6101978ab",
    "all_data_det2_s0.pt": "1b4037845e962540a26ebedca3f516c4cf0916ff77802967bd3730a52875eea5",
    "all_data_det2_s1.pt": "5e9fa2627f884bf834ad727e8aa6b57bcf1034f7074c3d097d32f3faa234e2d7",
    "all_data_det4_s2.pt": "37b2b9bca53be9a80f0e6f4eb06fba851811c30cc92a14a2b4bd10f47ea3e352",
    "all_data_det4_s3.pt": "3bd60d2d0d72b3f34530771236405d8fe66f1e636d7b36fa6776f706faec98fc",
    "submission_validated.csv": "c5364b0501278716fb7876eaa8b00fcc314e267c952e78cecb3a755b7a96a79d",
    "submission_all_data.csv": "5f8d5d2e9a8cda9d7dd2ace12a0b36bf7d8a5daebb6f69e9457b2ed67bf69c19"
}


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


for entry in ENTRIES_TO_RUN:
    spec = ENTRIES[entry]
    for name in spec["unets"] + [weights for weights, _ in spec["detectors"]] + [spec["submitted"]]:
        actual = sha256(WEIGHTS / name)
        if actual != SHA256[name]:
            raise SystemExit(f"{name}: sha256 {actual[:12]}, expected {SHA256[name][:12]}")
print("every weight file matches its recorded SHA-256")

every weight file matches its recorded SHA-256


## 1. Preprocessing

`scripts/preprocess.py` fits the solar limb in every frame (the bright halo beyond it makes a plain threshold overshoot by a median 39 pixels) and divides out limb darkening, caching a corrected 8-bit image and the disk geometry per observation. With validation on it processes the 707 training and 180 test observations, about 8 minutes on Kaggle's 4 CPUs; without, only the test frames. `scripts/make_splits.py` then rebuilds the train/validation split, which groups observations by date so no day is on both sides.

In [4]:
def run(name: str, *argv) -> None:
    """Run one repository script from the repository root, logging to LOGS/<name>.log."""
    started = time.time()
    log_path = LOGS / f"{name}.log"
    with open(log_path, "w") as log:
        code = subprocess.run([str(a) for a in argv], cwd=REPO, stdout=log,
                              stderr=subprocess.STDOUT).returncode
    tail = [line for line in log_path.read_text().splitlines() if line.strip() and "Warning" not in line]
    print(f"{name}: {'done' if code == 0 else 'FAILED'} in {(time.time() - started) / 60:.1f} min")
    for line in tail[-3:]:
        print("   ", line)
    if code:
        raise RuntimeError(f"{name} failed; see {log_path}")


# Without validation only the test frames are needed: an empty training folder skips the rest.
no_train = WORK / "no_training_frames"
no_train.mkdir(exist_ok=True)
run("preprocess", sys.executable, "scripts/preprocess.py", "--cache-dir", CACHE,
    "--workers", os.cpu_count() or 2, *([] if VALIDATE else ["--train-images", no_train]))
run("make_splits", sys.executable, "scripts/make_splits.py", "--group-by", "date")

from filament_seg.config import SPLIT_PATH, TEST_IMAGE_DIR, TRAIN_ANNOTATIONS
from filament_seg.data import load_annotations, load_split, stems_of, test_image_paths

TEST_STEMS = sorted(p.stem for p in test_image_paths(TEST_IMAGE_DIR))[: LIMIT or None]
split = load_split(SPLIT_PATH)
annotations = load_annotations(TRAIN_ANNOTATIONS)
n_train, n_val = (len(stems_of(annotations, split[part])) for part in ("train", "val"))
print(f"{len(TEST_STEMS)} test observations; split: {n_train} train, {n_val} validation observations")
assert (n_train, n_val) == (563, 144), "not the split the entries were validated on"

preprocess: done in 7.8 min
    wrote 887 disk geometries -> /tmp/reproduce/cache/disk.json
    wrote 1154 new mask PNGs -> /tmp/reproduce/cache/mask
    wrote 707 new consensus PNGs -> /tmp/reproduce/cache/consensus
make_splits: done in 0.0 min
    val  : 248 views / 144 observations
    observation overlap between train and val: 0 (must be 0)
    wrote /tmp/reproduce/outputs/splits.json
180 test observations; split: 563 train, 144 validation observations


## 2. U-Net logits and detections

Every model is one job: `scripts/cache_logits.py` caches a U-Net's logits (512-pixel tiles overlapping by 128, blended with a Hann window, averaged over the 8 flips and rotations), and `scripts/yolo_predict.py` saves a detector's instances (confidence ≥ 0.05, NMS IoU 0.6, at most 150 per image, full-resolution masks), the same calls the Kaggle detector runs made. Jobs run one per GPU at a time, longest first; each logs to `WORK/logs`. A job whose output already exists is skipped, so a rerun continues where it stopped.

In [5]:
import importlib.util

SLOTS = [str(i) for i in range(torch.cuda.device_count())] or [None]
YOLO_DEVICE = "0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


def run_jobs(jobs: list[tuple[str, list, Path | None]]) -> None:
    """Run (name, argv, output) jobs, at most one per GPU at a time; skip a job whose output exists."""
    pending = [job for job in jobs if not (job[2] and job[2].exists())]
    print(f"{len(jobs) - len(pending)} of {len(jobs)} jobs already done; "
          f"running {len(pending)} on {len(SLOTS)} {'GPU' if SLOTS[0] is not None else 'device'}(s)")
    running, failed = {}, []
    while pending or running:
        for slot in SLOTS:
            if slot not in running and pending:
                name, argv, _ = pending.pop(0)
                env = dict(os.environ)
                if slot is not None:
                    env["CUDA_VISIBLE_DEVICES"] = slot
                log = open(LOGS / f"{name}.log", "w")
                process = subprocess.Popen([str(a) for a in argv], cwd=REPO, env=env,
                                           stdout=log, stderr=subprocess.STDOUT)
                running[slot] = (name, process, log, time.time())
        time.sleep(2)
        for slot, (name, process, log, started) in list(running.items()):
            if process.poll() is None:
                continue
            log.close()
            del running[slot]
            minutes = (time.time() - started) / 60
            if process.returncode:
                failed.append(name)
                print(f"  {name}: FAILED after {minutes:.1f} min, see {LOGS / name}.log", flush=True)
            else:
                print(f"  {name}: done in {minutes:.1f} min", flush=True)
    if failed:
        raise RuntimeError(f"{len(failed)} job(s) failed: {failed}")


def unet_job(weights: str, subset: str):
    out = LOGITS / f"{Path(weights).stem}_{subset}"
    argv = [sys.executable, "scripts/cache_logits.py", "--checkpoint", WEIGHTS / weights,
            "--subset", subset, "--out-dir", out, "--cache-dir", CACHE]
    if subset == "test" and LIMIT:
        argv += ["--limit", LIMIT]
    # The cache skips maps it already holds, so this job is never skipped as a whole.
    return (f"unet_{Path(weights).stem}_{subset}", argv, None), out


def detector_job(weights: str, imgsz: int, subset: str):
    out = DETECTIONS / f"{Path(weights).stem}_{subset}{TAG if subset == 'test' else ''}.json"
    argv = [sys.executable, "scripts/yolo_predict.py", "--weights", WEIGHTS / weights,
            "--subset", subset, "--imgsz", imgsz, "--device", YOLO_DEVICE,
            "--cache-dir", CACHE, "--out", out]
    if subset == "test" and LIMIT:
        argv += ["--limit", LIMIT]
    return (f"det_{Path(weights).stem}_{subset}", argv, out), out


wanted = [(entry, "test") for entry in ENTRIES_TO_RUN] + ([("validated", "val")] if VALIDATE else [])
unet_jobs, detector_jobs, LOGIT_DIRS, DETECTION_FILES = [], [], {}, {}
for entry, subset in wanted:
    for weights in ENTRIES[entry]["unets"]:
        job, LOGIT_DIRS[weights, subset] = unet_job(weights, subset)
        unet_jobs.append(job)
    for weights, imgsz in ENTRIES[entry]["detectors"]:
        job, DETECTION_FILES[weights, subset] = detector_job(weights, imgsz, subset)
        detector_jobs.append(job)

if any(not out.exists() for _, _, out in detector_jobs) and importlib.util.find_spec("ultralytics") is None:
    raise SystemExit("the detectors need ultralytics: pip install -r requirements-detector.txt")
run_jobs(unet_jobs + detector_jobs)

0 of 18 jobs already done; running 18 on 2 GPU(s)
  unet_all_data_unet_s0_test: done in 12.7 min
  unet_validated_unet_run3_last_test: done in 12.9 min
  unet_all_data_unet_s1_test: done in 12.7 min
  unet_all_data_unet_s2_test: done in 12.6 min
  unet_validated_unet_run3_last_val: done in 10.0 min
  det_validated_det1_yolo11s_1024_best_test: done in 1.7 min
  unet_all_data_unet_s3_test: done in 12.8 min
  det_validated_det2_yolo11m_1280_best_test: done in 1.9 min
  det_validated_det4_yolo11m_1536_last_test: done in 2.1 min
  det_all_data_det1_s0_test: done in 1.7 min
  det_all_data_det1_s1_test: done in 1.6 min
  det_all_data_det2_s0_test: done in 1.8 min
  det_all_data_det2_s1_test: done in 1.9 min
  det_all_data_det4_s2_test: done in 2.0 min
  det_all_data_det4_s3_test: done in 2.1 min
  det_validated_det1_yolo11s_1024_best_val: done in 1.4 min
  det_validated_det2_yolo11m_1280_best_val: done in 1.6 min
  det_validated_det4_yolo11m_1536_last_val: done in 1.7 min


## 3. Ensemble and merge

The all-data entry averages its four U-Nets' logits (`scripts/average_logits.py`, equal weights, exactly what an ensemble forward pass computes). Each entry's detections are merged across its detectors (`scripts/merge_detections.py`): instances that overlap with IoU above 0.5 are one filament, keep the most confident mask, and score the mean confidence over the entry's detectors, a detector that missed the filament counting as 0.

In [6]:
import shutil

from filament_seg.logit_cache import cache_path


def entry_logits(entry: str, subset: str) -> Path:
    members = [LOGIT_DIRS[weights, subset] for weights in ENTRIES[entry]["unets"]]
    if len(members) == 1:
        return members[0]
    out = LOGITS / f"{entry}_ensemble_{subset}{TAG}"
    stems = TEST_STEMS if subset == "test" else stems_of(annotations, split["val"])
    if not all(cache_path(out, stem).exists() for stem in stems):
        shutil.rmtree(out, ignore_errors=True)  # an interrupted average: start it again
        run(f"average_{entry}_{subset}", sys.executable, "scripts/average_logits.py", "--out", out, *members)
    return out


def entry_detections(entry: str, subset: str) -> Path:
    out = DETECTIONS / f"{entry}_merged_{subset}{TAG if subset == 'test' else ''}.json"
    inputs = [DETECTION_FILES[weights, subset] for weights, _ in ENTRIES[entry]["detectors"]]
    run(f"merge_{entry}_{subset}", sys.executable, "scripts/merge_detections.py", "--out", out, *inputs)
    return out


ENTRY_LOGITS = {entry: entry_logits(entry, "test") for entry in ENTRIES_TO_RUN}
ENTRY_DETECTIONS = {entry: entry_detections(entry, "test") for entry in ENTRIES_TO_RUN}

average_all_data_test: done in 0.2 min
    wrote /tmp/reproduce/logits/all_data_ensemble_test: 180 observations averaged over 4 models
merge_validated_test: done in 0.0 min
    wrote /tmp/reproduce/detections/validated_merged_test.json: 4119 detections over 180 observations from 3 detectors
merge_all_data_test: done in 0.0 min
    wrote /tmp/reproduce/detections/all_data_merged_test.json: 4655 detections over 180 observations from 6 detectors


## 4. Fusion

For each test image: the U-Net probability is thresholded at 0.5, closed with a 3-pixel disk, hole-filled and restricted to the solar disk; then, in descending confidence, each detection scoring at least 0.30 claims the unclaimed foreground inside its mask grown by 4 pixels; claims under 200 pixels and foreground no detection claims are dropped. Instances never overlap. The result is written in the competition's submission format.

In [7]:
import numpy as np

from filament_seg.dataset import load_disk_geometry
from filament_seg.fusion import Detection, FusionParams, fuse
from filament_seg.rle import build_submission, labels_to_rles, write_submission

disk_geometry = load_disk_geometry(CACHE / "disk.json")
params = FusionParams(**FUSION)
SUBMISSIONS = {}
for entry in ENTRIES_TO_RUN:
    started = time.time()
    raw = json.loads(ENTRY_DETECTIONS[entry].read_text(encoding="utf-8"))
    detections = {stem: [Detection(d["score"], d["counts"]) for d in found] for stem, found in raw.items()}
    predictions = {}
    for stem in TEST_STEMS:
        logits = np.load(cache_path(ENTRY_LOGITS[entry], stem)).astype(np.float32)
        labels = fuse(logits, disk_geometry[stem].mask(logits.shape), detections.get(stem, []), params)
        predictions[stem] = labels_to_rles(labels)
    frame = build_submission(predictions)
    SUBMISSIONS[entry] = write_submission(frame, RESULTS / f"submission_{entry}{TAG}.csv")
    print(f"{entry}: {len(frame)} instances over {len(predictions)} images in "
          f"{time.time() - started:.0f} s -> {SUBMISSIONS[entry]}")

validated: 1097 instances over 180 images in 56 s -> /kaggle/working/submission_validated.csv
all_data: 1088 instances over 180 images in 56 s -> /kaggle/working/submission_all_data.csv


## 5. Comparison with the submitted files

There are no test labels, so the rebuilt file is scored against the submitted one as if that were the ground truth (`scripts/agreement.py`'s measure): agreement PQ 1.0 means the same instances with the same masks. An image counts as identical when its instance masks match exactly.

In [8]:
from filament_seg.metrics import evaluate_image, summarize
from filament_seg.rle import counts_of, read_submission

for entry in ENTRIES_TO_RUN:
    rebuilt = read_submission(SUBMISSIONS[entry])
    submitted = read_submission(WEIGHTS / ENTRIES[entry]["submitted"])
    identical = sum(sorted(map(counts_of, rebuilt.get(stem, []))) == sorted(map(counts_of, submitted.get(stem, [])))
                    for stem in TEST_STEMS)
    summary = summarize([evaluate_image(stem, submitted.get(stem, []), rebuilt.get(stem, []),
                                        with_fragmentation=False) for stem in TEST_STEMS])
    n_rebuilt = sum(len(rebuilt.get(stem, [])) for stem in TEST_STEMS)
    n_submitted = sum(len(submitted.get(stem, [])) for stem in TEST_STEMS)
    print(f"{entry}: {identical}/{len(TEST_STEMS)} images identical; {n_rebuilt} instances rebuilt, "
          f"{n_submitted} submitted; agreement PQ {summary['pq_pooled']:.4f} "
          f"(SQ {summary['sq']:.4f}, RQ {summary['rq']:.4f})")

validated: 174/180 images identical; 1097 instances rebuilt, 1097 submitted; agreement PQ 0.9999 (SQ 0.9999, RQ 1.0000)
all_data: 178/180 images identical; 1088 instances rebuilt, 1088 submitted; agreement PQ 1.0000 (SQ 1.0000, RQ 1.0000)


## 6. Validation of the validated entry

`scripts/fuse_detections.py` scores the validated entry on the 144 validation observations as the competition scores the test set: each annotator's view separately, true positives at IoU > 0.5, totals pooled over all views. It is cross-fitted: the observations are split into two halves of 72 (`sample_stems`, seed 0), each half is scored with the fusion setting that is best on the other half, and so every observation is out of sample. The report gives 0.4214 (Table 1), against 0.371 for the U-Net alone. The setting that is best on all 144 observations is the one the submissions use.

In [9]:
if VALIDATE:
    unet = ENTRIES["validated"]["unets"][0]
    started = time.time()
    run("fuse_validation", sys.executable, "scripts/fuse_detections.py",
        "--checkpoint", WEIGHTS / unet, "--cache-dir", CACHE, "--logit-dir", LOGIT_DIRS[unet, "val"],
        "--detections-val", entry_detections("validated", "val"),
        "--baseline", "threshold=0.6,min_area=400,bridge_gap=24,close_radius=3,open_radius=0",
        "--threshold", 0.5, 0.6, 0.7, "--grow", 4, 8, 16, "--min-score", 0.2, 0.25, 0.3, 0.35,
        "--min-area", 200, 400, "--keep-unclaimed", 0,
        "--out", RESULTS / "validation_validated.json", "--save-totals", RESULTS / "totals_validated.npz")
    result = json.loads((RESULTS / "validation_validated.json").read_text(encoding="utf-8"))
    crossfit = result["crossfit"]
    print(f"cross-fitted PQ {crossfit['pq']:.4f} (report: 0.4214); vs U-Net only "
          f"{crossfit['delta_vs_baseline']:+.4f} [{crossfit['ci_low']:+.4f}, {crossfit['ci_high']:+.4f}]")
    print(f"best setting on all 144 observations: {result['best']}")
else:
    print("skipped (VALIDATE is off)")

if RESULTS != WORK:
    shutil.copytree(LOGS, RESULTS / "logs", dirs_exist_ok=True)
print(f"\nnotebook total: {(time.time() - NOTEBOOK_START) / 60:.0f} min")

merge_validated_val: done in 0.0 min
    wrote /tmp/reproduce/detections/validated_merged_val.json: 3402 detections over 144 observations from 3 detectors
fuse_validation: done in 11.4 min
      tuned on the held-out half: fusion thr=0.50 close=3 grow=4 score>=0.30 area=200 keep=0/24
    wrote /kaggle/working/validation_validated.json
    wrote /kaggle/working/totals_validated.npz (compare runs with scripts/compare_runs.py)
cross-fitted PQ 0.4214 (report: 0.4214); vs U-Net only +0.0506 [+0.0370, +0.0642]
best setting on all 144 observations: fusion thr=0.50 close=3 grow=4 score>=0.30 area=200 keep=0/24
notebook total: 70 min


## 7. Training (not rerun here)

Each model was trained in a private Kaggle session (2× Tesla T4, 4 CPUs) that cloned the repository at the commit listed below and ran the same setup first:

```bash
git clone https://github.com/ameyypawar/filament-seg && cd filament-seg && git checkout <commit>
pip install segmentation-models-pytorch "ultralytics>=8.3,<9"
export FILAMENT_DATA_ROOT=<competition data> FILAMENT_OUTPUT_ROOT=<outputs>
python scripts/preprocess.py --workers 4        # 887 observations, 7-8.5 min
python scripts/make_splits.py --group-by date   # 563 training / 144 validation observations
```

Detector sessions then export the training views in YOLO format: `python scripts/export_yolo.py --out <yolo_dir>` for the validated entry (906 annotator views), with `--all-data` for the all-data entry (1,154 views; Ultralytics requires a validation set, so 16 views are listed as one, unused under `--no-val`). Every annotator view is its own sample.

| Weights file | Commit | Session (h) | Training command |
|---|---|---|---|
| `validated_unet_run3_last.pt` | `378f8af` | 2.39 | `python scripts/train.py --epochs 30 --batch-size 16 --workers 4 --limit-val 48 --target annotator --out model_best.pt`; the last epoch is saved as `model_best_last.pt` |
| `validated_det1_yolo11s_1024_best.pt` | `68dc491` | 0.93 | `python scripts/train_detector.py --data <yolo_dir>/data.yaml --out detector --model yolo11s-seg.pt --imgsz 1024 --epochs 60 --batch 8 --device 0 --workers 4` (seed 0, patience 20; best epoch) |
| `validated_det2_yolo11m_1280_best.pt` | `5817c4b`* | not recorded* | `python scripts/train_detector.py --data <yolo_dir>/data.yaml --out detector --model yolo11m-seg.pt --imgsz 1280 --epochs 80 --batch 8 --device 0,1 --workers 4 --patience 25` (seed 0; best epoch) |
| `validated_det4_yolo11m_1536_last.pt` | `3061d9c` | 2.84 | `python scripts/train_detector.py --data <yolo_dir>/data.yaml --out detector --model yolo11m-seg.pt --imgsz 1536 --epochs 80 --batch 8 --device 0,1 --workers 4 --seed 2 --patience 25` (last epoch) |
| `all_data_unet_s0.pt`, `all_data_unet_s1.pt` | `3061d9c` | 1.52 for both | one per GPU at the same time: `CUDA_VISIBLE_DEVICES=0 python scripts/train.py --epochs 30 --batch-size 16 --workers 2 --limit-val 48 --preload --out model_alldata_s0.pt --all-data`, and the same on GPU 1 with `--seed 1` |
| `all_data_unet_s2.pt`, `all_data_unet_s3.pt` | `3061d9c` | 1.59 for both | as above with `--seed 2` and `--seed 3` |
| `all_data_det1_s0.pt`, `all_data_det1_s1.pt` | `3061d9c` | 1.02, 1.02 | `python scripts/train_detector.py --data <yolo_dir>/data.yaml --out detector --model yolo11s-seg.pt --imgsz 1024 --epochs 60 --batch 8 --device 0 --workers 4 --seed 0 --patience 0 --no-val` (`--seed 1` for the second) |
| `all_data_det2_s0.pt`, `all_data_det2_s1.pt` | `3061d9c` | 2.30, 2.02 | the same with `--model yolo11m-seg.pt --imgsz 1280 --epochs 80 --device 0,1` and `--seed 0` / `--seed 1` |
| `all_data_det4_s2.pt`, `all_data_det4_s3.pt` | `3061d9c` | 3.22, 3.13 | the same with `--model yolo11m-seg.pt --imgsz 1536 --epochs 80 --device 0,1` and `--seed 2` / `--seed 3` |

\* Detector 2's session left no log. Its settings are those of the `args.yaml` Ultralytics saved with it; its commit is the one on GitHub when it was trained (2 October), whose training scripts are identical to `68dc491`'s; Ultralytics' own clock gave 1.73 h of training. The recorded sessions total 6.2 h for the validated entry (without detector 2) and 15.8 h for the all-data entry.

**U-Net.** segmentation_models_pytorch U-Net with an ImageNet-initialised ResNet34 encoder and two input channels (corrected intensity, radial distance normalised to the limb); 512-pixel crops at native resolution, 8 per observation per epoch; AdamW at 1e-3 with a cosine schedule; soft Dice plus binary cross-entropy; mixed precision. `--all-data` trains on the training and validation observations together, with no validation; the checkpoint is the last epoch.

**Detectors.** Ultralytics YOLO11-seg from COCO weights, one class, with the augmentation fixed in `scripts/train_detector.py` (mosaic off, flips, small translations and scaling, brightness jitter); `optimizer=auto` resolved to AdamW at 0.002.

**Versions.** PyTorch 2.10.0 (2.11.0 for `all_data_det1_s1`, `all_data_det4_s2` and `all_data_det4_s3`), Ultralytics 8.4.171 to 8.4.173.

Retraining reproduces the recipes, not the exact weights: Ultralytics trains with `deterministic=False`, and the U-Net's `--seed` fixes which annotator view and which crops it sees, not the GPU arithmetic.

## Notes on determinism

The submitted files were made with U-Net logits computed on an Apple M-series GPU (MPS) and detections computed on Kaggle T4 GPUs. Other hardware rounds the floating-point arithmetic differently in the last digits, so a rebuilt mask can differ from the submitted one by a few pixels. The run above (two T4s, PyTorch 2.11) compares with the submitted files as follows:

| | validated entry | all-data entry |
|---|---|---|
| images identical to the submitted file | 174 of 180 | 178 of 180 |
| instances matched at IoU > 0.5 | 1,097 of 1,097 | 1,088 of 1,088 |
| instances that differ | 7, by 1 to 57 pixels | 2, by 1 and 3 pixels |
| agreement PQ | 0.99994 | 0.999997 |

The validation score reproduces too: cross-fitted PQ 0.421444 against the report's 0.421443, with the same best fusion setting. On the Mac that made the submissions, with the original detections, the notebook rebuilds both files exactly (checked on the first six test images).